# Custom Batch Strategy

PromptForge ships with two built-in batch strategies:

| Strategy | When to use |
|---|---|
| `RandomBatchStrategy` (default) | General purpose; ensures every example is eventually seen |
| `FailurePriorityBatchStrategy` | Focuses on examples the current prompt gets wrong |

This notebook shows how to implement a **custom** `BatchStrategy` by subclassing the abstract base class. We build two examples:

1. **`SequentialBatchStrategy`** — iterates through examples in a fixed order (useful for debugging or reproducibility).
2. **`TokenBudgetBatchStrategy`** — selects as many examples as fit within a token budget, skipping large examples.

## 1 — The `BatchStrategy` interface

Every strategy must implement a single method:

```python
def select_batch(
    self,
    bundles: list[ExampleBundle],
    batch_size: int,
    used_ids: set[str],
    failed_ids: set[str] | None = None,
) -> list[ExampleBundle]:
    ...
```

- **`bundles`** — all available training examples.
- **`batch_size`** — how many to return (treat as a *maximum*, not a hard requirement).
- **`used_ids`** — bundle IDs that have already been trained on across all previous iterations.
- **`failed_ids`** — bundle IDs that the current prompt got wrong (may be `None` or empty).

In [ ]:
from prompt_forge.training.batch_strategy import BatchStrategy
from prompt_forge.bundle import ExampleBundle


class SequentialBatchStrategy(BatchStrategy):
    """
    Iterates through examples in a fixed (index) order.

    Once all examples have been used, it cycles back to the beginning.
    This is deterministic and useful for debugging.
    """

    def select_batch(
        self,
        bundles: list[ExampleBundle],
        batch_size: int,
        used_ids: set[str],
        failed_ids: set[str] | None = None,
    ) -> list[ExampleBundle]:
        # Prefer examples not yet used
        unused = [b for b in bundles if b.bundle_id not in used_ids]
        pool = unused if unused else bundles  # cycle when exhausted
        return pool[:batch_size]

## 2 — Token-budget batch strategy

When examples vary a lot in size (e.g. short invoices vs. 50-page contracts) it can be useful to build a batch that fits within a token budget rather than a fixed count.

The strategy below uses a simple character-based estimator (≈ 4 chars per token). You can swap in any tokeniser.

In [ ]:
import random
from prompt_forge.file_loaders import get_default_loader


class TokenBudgetBatchStrategy(BatchStrategy):
    """
    Selects examples that fit within a token budget.

    Examples that individually exceed the budget are silently skipped.
    The batch is filled greedily in random order until the budget is exhausted.

    Args:
        token_budget: Maximum *estimated* tokens for the selected batch.
        chars_per_token: Characters-per-token ratio used for estimation (default 4).
    """

    def __init__(self, token_budget: int, chars_per_token: int = 4):
        self.token_budget = token_budget
        self.chars_per_token = chars_per_token
        self._loader = get_default_loader()

    def _estimate_tokens(self, bundle: ExampleBundle) -> int:
        try:
            contents = bundle.load_contents(self._loader)
            total_chars = sum(len(c.text) for c in contents.values())
            return total_chars // self.chars_per_token
        except Exception:
            return 0  # skip unreadable examples rather than crash

    def select_batch(
        self,
        bundles: list[ExampleBundle],
        batch_size: int,
        used_ids: set[str],
        failed_ids: set[str] | None = None,
    ) -> list[ExampleBundle]:
        # Prioritise unused examples; shuffle for variety
        unused = [b for b in bundles if b.bundle_id not in used_ids]
        pool = unused if unused else list(bundles)
        random.shuffle(pool)

        selected = []
        remaining_budget = self.token_budget

        for bundle in pool:
            if len(selected) >= batch_size:
                break
            cost = self._estimate_tokens(bundle)
            if cost <= remaining_budget:
                selected.append(bundle)
                remaining_budget -= cost

        return selected

## 3 — Using a custom strategy in training

Pass the strategy instance to `Project.train()` via the `batch_strategy` parameter.

In [ ]:
# ── Replace with your real LLM client ─────────────────────────────────────────
# from openai import AzureOpenAI
# from prompt_forge.llm.openai_client import OpenAIClient
# llm = OpenAIClient(AzureOpenAI(...), model="gpt-4o")

# ── Demo: use a mock LLM so the notebook runs without credentials ──────────────
from prompt_forge.llm.client import LLMClient, LLMMessage, LLMResponse

class MockLLM(LLMClient):
    def complete(self, messages: list[LLMMessage], **kwargs) -> LLMResponse:
        return LLMResponse(
            text="IMPROVED PROMPT (mock)",
            usage={"input_tokens": 100, "output_tokens": 50},
        )

llm = MockLLM()

In [ ]:
from prompt_forge import Project

project = Project("demo", llm=llm, project_dir="/tmp/demo_project")

# Set up a minimal schema and seed prompt
project.set_bundle_schema(input=".txt", expected_output=".txt")
project.set_seed_prompt("Summarise the input text concisely.")

# NOTE: add real examples before running train()
# project.add_examples_from_directory("./data")

print("Project ready:", project)

In [ ]:
# ── Sequential strategy ────────────────────────────────────────────────────────
# report = project.train(
#     batch_size=4,
#     max_iterations=10,
#     eval_strategy=None,          # no evaluation — run all iterations
#     extract_learnings=False,     # skip the second LLM call to save tokens
#     batch_strategy=SequentialBatchStrategy(),
# )

# ── Token-budget strategy ──────────────────────────────────────────────────────
# report = project.train(
#     batch_size=10,
#     max_iterations=10,
#     eval_strategy=None,
#     batch_strategy=TokenBudgetBatchStrategy(token_budget=8_000),
# )

print("Uncomment one of the train() calls above after loading real examples.")

## 4 — Tips

- **`used_ids`** tracks every bundle seen across *all* iterations. Prefer `unused` examples first so the model sees diverse data before revisiting.
- **`failed_ids`** is only populated when using `FailurePriorityBatchStrategy` internally; your strategy can ignore it or use it to up-weight hard examples.
- You can combine strategies: start with `SequentialBatchStrategy` for the first pass, then switch to `FailurePriorityBatchStrategy` for subsequent passes.
- The `batch_size` parameter is a *suggestion*. It is fine to return fewer examples (e.g. when the token budget is tight) — the pipeline will not error.